# 00 — Synthetic Marketing Data Generation
**Superstore Project | Business Statistics & Insights | Master in Business Analytics & AI**

<img src="https://quivo.co/fileadmin/_processed_/8/b/csm_Online-Shop_big_e8834fc4db.jpg" width="1500">
---

## Purpose
The Superstore Sales dataset records transactions (sales, profit, discount) but contains no marketing channel data — there is no record of *how* customers were acquired, *what* was spent to acquire them, or *which channels* drove conversion.

To demonstrate **Marketing Mix Profitability & Budget Allocation** skills (CLV, CAC, channel profitability, ROAS, incrementality), synthetic marketing variables were created with **explicit approval from the course professor**.

This notebook is the single source of truth for all synthetic data. Every assumption is documented, every parameter is justified, and all outputs are validated before saving.

---

## Outputs produced
| File | Grain | Rows | Description |
|------|-------|------|-------------|
| `customer_acquisition.parquet/.csv` | 1 row per customer | 793 | Channel, CAC, campaign per customer |
| `monthly_channel_spend.parquet/.csv` | 1 row per channel × month | 288 | Spend, impressions, clicks, ROAS |
| `superstore_enriched.parquet/.csv` | 1 row per transaction | 9,994 | Original data + channel attribution |

---

## Reproducibility
All random processes use `numpy.random.seed(42)`. Re-running this notebook from top to bottom produces identical outputs every time.

## 0. Imports and base dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)   # ← global seed — all synthetic randomness flows from here

# Load the cleaned transaction-level dataset (produced by 01_data_preparation.ipynb)
df = pd.read_parquet('../data/processed/superstore_clean.parquet')

print(f"Base dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Customers  : {df['Customer ID'].nunique():,}")
print(f"Segments   : {df['Segment'].value_counts().to_dict()}")

---
## Table 1 — `customer_acquisition`
**Grain:** one row per customer (793 rows)

### What it represents
Each customer has a single acquisition event — the channel through which they first discovered and converted on Superstore. In a real system this would come from a CRM, CDP (e.g. Twilio Segment), or last-click attribution model. Here it is simulated.

### Assumptions & justifications

**Channel assignment probabilities** are set per segment to reflect realistic B2C vs. B2B acquisition patterns:

| Channel | Consumer | Corporate | Home Office | Rationale |
|---------|----------|-----------|-------------|-----------|
| Paid Search | 28% | 30% | 22% | High intent, works across all segments |
| Social Media | 32% | 5% | 25% | Strong for consumer; negligible in B2B |
| Email | 10% | 32% | 15% | Dominant B2B channel |
| Organic | 18% | 12% | 28% | Home Office buyers research independently |
| Display | 8% | 3% | 6% | Upper-funnel, lower conversion |
| Direct | 4% | 18% | 4% | Sales team / referrals matter more in Corporate |

**CAC (Customer Acquisition Cost)** benchmarks are based on industry averages for retail/B2B SaaS:

| Channel | Mean CAC | Std Dev | Source rationale |
|---------|----------|---------|-----------------|
| Paid Search | $28 | $12 | Google Ads retail benchmarks |
| Social Media | $22 | $9 | Meta Ads e-commerce averages |
| Email | $6 | $3 | Low cost — list-based, minimal media spend |
| Organic | $10 | $5 | SEO/content investment amortised per customer |
| Display | $38 | $15 | Programmatic CPM — high spend, lower CVR |
| Direct | $3 | $2 | Referral / word-of-mouth — near-zero media cost |

In [ ]:
# ── Step 1.1 — Build customer-level base table ──────────────────────────────
customers = df.groupby('Customer ID').agg(
    Customer_Name  = ('Customer Name', 'first'),
    Segment        = ('Segment',       'first'),
    Region         = ('Region',        'first'),
    First_Order    = ('Order Date',    'min'),
    Total_Orders   = ('Order ID',      'nunique'),
    Total_Revenue  = ('Sales',         'sum'),
    Total_Profit   = ('Profit',        'sum'),
).reset_index()

print(f"Customer-level table: {len(customers)} rows")
print(customers[['Total_Orders','Total_Revenue','Total_Profit']].describe().round(2))

In [ ]:
# ── Step 1.2 — Assign acquisition channel (probability-weighted per segment) ──
channel_probs = {
    'Consumer':    {'Paid Search': 0.28, 'Social Media': 0.32, 'Email': 0.10,
                    'Organic':     0.18, 'Display':      0.08, 'Direct': 0.04},
    'Corporate':   {'Paid Search': 0.30, 'Social Media': 0.05, 'Email': 0.32,
                    'Organic':     0.12, 'Display':      0.03, 'Direct': 0.18},
    'Home Office': {'Paid Search': 0.22, 'Social Media': 0.25, 'Email': 0.15,
                    'Organic':     0.28, 'Display':      0.06, 'Direct': 0.04},
}

# Validate: probabilities must sum to 1 per segment
for seg, probs in channel_probs.items():
    assert abs(sum(probs.values()) - 1.0) < 1e-9, f"Probabilities don't sum to 1 for {seg}"
print("✓ Channel probabilities validated (sum = 1.0 per segment)")

channels = []
for _, row in customers.iterrows():
    probs = channel_probs[row['Segment']]
    ch    = np.random.choice(list(probs.keys()), p=list(probs.values()))
    channels.append(ch)

customers['Acquisition_Channel'] = channels

print("\nChannel distribution:")
print(customers['Acquisition_Channel'].value_counts())
print("\nChannel × Segment cross-tab (%):")
print(pd.crosstab(customers['Segment'], customers['Acquisition_Channel'],
                  normalize='index').mul(100).round(1))

In [ ]:
# ── Step 1.3 — Generate CAC per customer (Normal distribution, clipped at $1) ──
cac_params = {
    'Paid Search':  (28, 12),
    'Social Media': (22,  9),
    'Email':        ( 6,  3),
    'Organic':      (10,  5),
    'Display':      (38, 15),
    'Direct':       ( 3,  2),
}

cacs = []
for ch in customers['Acquisition_Channel']:
    mu, sd = cac_params[ch]
    cost   = max(1.0, round(np.random.normal(mu, sd), 2))  # floor at $1
    cacs.append(cost)

customers['CAC'] = cacs

print("CAC statistics:")
print(customers.groupby('Acquisition_Channel')['CAC'].agg(['mean','median','min','max']).round(2))

In [ ]:
# ── Step 1.4 — Assign acquisition campaign (channel-specific) ───────────────
campaign_map = {
    'Paid Search':  ['Brand_Awareness_PPC', 'Competitor_Keywords', 'Product_Category_PPC'],
    'Social Media': ['Facebook_Prospecting', 'Instagram_Retargeting', 'LinkedIn_B2B'],
    'Email':        ['Welcome_Series', 'Promotional_Blast', 'Re_Engagement'],
    'Organic':      ['SEO_Content', 'Blog_Traffic', 'Referral_Program'],
    'Display':      ['Retargeting_Display', 'Prospecting_Display'],
    'Direct':       ['Word_of_Mouth', 'Sales_Outreach'],
}

customers['Acquisition_Campaign'] = customers['Acquisition_Channel'].apply(
    lambda ch: np.random.choice(campaign_map[ch])
)

print("Campaign distribution:")
print(customers['Acquisition_Campaign'].value_counts())

In [ ]:
# ── Step 1.5 — Compute LTV:CAC ratio ────────────────────────────────────────
# LTV proxy = Total_Profit (full observed lifetime in dataset)
# Benchmark: LTV:CAC ≥ 3 is considered healthy
customers['LTV_CAC_Ratio'] = (customers['Total_Profit'] / customers['CAC']).round(2)

print("LTV:CAC ratio summary:")
print(customers['LTV_CAC_Ratio'].describe().round(2))

below_benchmark = (customers['LTV_CAC_Ratio'] < 3).mean() * 100
print(f"\n{below_benchmark:.1f}% of customers are below the 3× LTV:CAC benchmark")

In [ ]:
# ── Step 1.6 — Validate and inspect final table ──────────────────────────────
assert customers.isnull().sum().sum() == 0, "Nulls found in customer_acquisition"
print(f"✓ No nulls in customer_acquisition")
print(f"✓ Rows: {len(customers)} | Columns: {len(customers.columns)}")
print(f"\nColumn list: {customers.columns.tolist()}")
print(f"\nSample (5 rows):")
customers[['Customer ID','Segment','Acquisition_Channel','CAC',
           'Acquisition_Campaign','LTV_CAC_Ratio']].head()

---
## Table 2 — `monthly_channel_spend`
**Grain:** one row per channel × month (6 channels × 48 months = 288 rows)

### What it represents
Monthly marketing budget allocation and resulting performance metrics per channel. In a real business this would come from Google Ads, Meta Ads Manager, and a marketing data warehouse via connectors like Fivetran.

### Assumptions & justifications

**Base monthly spend** reflects a realistic mid-market retailer budget:

| Channel | Base/Month | Rationale |
|---------|-----------|-----------|
| Paid Search | $12,000 | Largest paid channel — high intent, scalable |
| Social Media | $8,000 | Growing channel, especially visual categories |
| Display | $6,000 | Upper-funnel brand; less efficient per conversion |
| Organic | $3,500 | SEO/content team cost amortised monthly |
| Email | $2,500 | ESP + content; very low marginal cost |
| Direct | $1,500 | Sales team time amortised |

**Modifiers applied:**
- **Seasonal multiplier:** ×1.45 in Nov–Dec (Q4 peak), ×0.80 in Jan–Feb (Q1 dip), ×0.90 Jul–Aug (summer)
- **YoY growth:** Paid Search +12%/yr, Social Media +22%/yr (fastest growing), Display +3%/yr (declining)
- **Noise:** ±5% random variation per month to simulate real budget fluctuations

In [ ]:
# ── Step 2.1 — Define parameters ────────────────────────────────────────────
channels_list = ['Paid Search', 'Social Media', 'Email', 'Organic', 'Display', 'Direct']
months        = pd.period_range('2014-01', '2017-12', freq='M')   # 48 months

base_spend = {
    'Paid Search': 12000, 'Social Media': 8000, 'Email': 2500,
    'Organic': 3500, 'Display': 6000, 'Direct': 1500,
}
growth_rate = {
    'Paid Search': 0.12, 'Social Media': 0.22, 'Email': 0.05,
    'Organic': 0.08, 'Display': 0.03, 'Direct': 0.04,
}

def seasonal_multiplier(period):
    m = period.month
    if m in [11, 12]: return 1.45   # Q4 — holiday peak
    if m in [1,  2]:  return 0.80   # Q1 — post-holiday dip
    if m in [7,  8]:  return 0.90   # summer slowdown
    return 1.0

# Channel performance parameters (impressions / CTR / CVR)
cpm_map = {'Paid Search': 8, 'Social Media': 6, 'Email': 0, 'Organic': 0, 'Display': 4, 'Direct': 0}
ctr_map = {'Paid Search': 0.035, 'Social Media': 0.018, 'Email': 0.22,
           'Organic': 0.04, 'Display': 0.005, 'Direct': 0.60}
cvr_map = {'Paid Search': 0.032, 'Social Media': 0.018, 'Email': 0.045,
           'Organic': 0.025, 'Display': 0.010, 'Direct': 0.12}

print("Parameters set. Channels:", channels_list)
print(f"Date range: {months[0]} → {months[-1]}  ({len(months)} months)")

In [ ]:
# ── Step 2.2 — Generate monthly records ─────────────────────────────────────
records = []
for period in months:
    yr_offset = period.year - 2014   # years since baseline

    for ch in channels_list:
        # Spend = base × growth × seasonality × noise
        spend = round(
            base_spend[ch]
            * (1 + growth_rate[ch]) ** yr_offset
            * seasonal_multiplier(period)
            * np.random.normal(1.0, 0.05),   # ±5% noise
            2
        )

        # Impressions (only for paid channels with a CPM)
        cpm   = cpm_map[ch]
        impr  = int(spend / cpm * 1000) if cpm > 0 else 0

        # Clicks
        clicks = int(impr * ctr_map[ch]) if impr > 0 else int(spend * 0.5)

        # Conversions (Binomial-like with small Gaussian noise)
        convs = max(0, int(clicks * cvr_map[ch] + np.random.normal(0, 2)))

        # Revenue attributed (avg order value × conversions)
        rev_attr = round(convs * np.random.uniform(180, 280), 2)
        roas     = round(rev_attr / spend, 3) if spend > 0 else 0

        records.append({
            'Period':              str(period),
            'Year':                period.year,
            'Month':               period.month,
            'Quarter':             period.quarter,
            'Channel':             ch,
            'Spend':               spend,
            'Impressions':         impr,
            'Clicks':              clicks,
            'Conversions':         convs,
            'Revenue_Attributed':  rev_attr,
            'ROAS':                roas,
        })

spend_df = pd.DataFrame(records)
print(f"Generated: {len(spend_df)} rows × {len(spend_df.columns)} columns")
print(spend_df.head(6).to_string())

In [ ]:
# ── Step 2.3 — Validate ──────────────────────────────────────────────────────
assert len(spend_df) == len(channels_list) * len(months), "Row count mismatch"
assert spend_df.isnull().sum().sum() == 0, "Nulls found"
assert (spend_df['Spend'] > 0).all(), "Zero/negative spend values found"
assert (spend_df['ROAS'] >= 0).all(), "Negative ROAS found"
print("✓ All validation checks passed")
print(f"\nTotal spend 2014–2017: ${spend_df['Spend'].sum():,.0f}")
print(f"\nSpend by channel:")
print(spend_df.groupby('Channel')['Spend'].sum().sort_values(ascending=False).apply(lambda x: f'${x:,.0f}'))
print(f"\nAvg ROAS by channel:")
print(spend_df.groupby('Channel')['ROAS'].mean().sort_values(ascending=False).round(2))

In [ ]:
# ── Step 2.4 — Quick visualisation: spend trends ────────────────────────────
pivot = spend_df.pivot(index='Period', columns='Channel', values='Spend')

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Spend trend
for ch in channels_list:
    axes[0].plot(pivot.index, pivot[ch], marker='o', markersize=2,
                 linewidth=1.5, label=ch)
axes[0].set_title('Monthly Spend by Channel — 2014 to 2017', fontweight='bold')
axes[0].set_ylabel('Spend ($)')
axes[0].legend(bbox_to_anchor=(1.01, 1), fontsize=8)
axes[0].set_xticks(pivot.index[::6])
axes[0].set_xticklabels(list(pivot.index[::6]), rotation=45, fontsize=8)
axes[0].grid(True, alpha=0.3)

# ROAS trend
roas_pivot = spend_df.pivot(index='Period', columns='Channel', values='ROAS')
for ch in channels_list:
    axes[1].plot(roas_pivot.index, roas_pivot[ch], marker='o', markersize=2,
                 linewidth=1.5, label=ch)
axes[1].axhline(1, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='Break-even')
axes[1].set_title('Monthly ROAS by Channel', fontweight='bold')
axes[1].set_ylabel('ROAS')
axes[1].legend(bbox_to_anchor=(1.01, 1), fontsize=8)
axes[1].set_xticks(roas_pivot.index[::6])
axes[1].set_xticklabels(list(roas_pivot.index[::6]), rotation=45, fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../output/images', exist_ok=True)
plt.savefig('../output/images/00_spend_roas_trends.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Table 3 — `superstore_enriched`
**Grain:** one row per transaction (9,994 rows)

### What it represents
The original transaction-level dataset enriched with channel attribution columns from Table 1. This is the primary analytical table for all channel-level profitability analysis.

### Join logic
- `LEFT JOIN customer_acquisition ON Customer ID`
- All 9,994 rows are retained (no transactions are lost)
- Each transaction inherits the acquisition channel and CAC of the customer who placed the order
- `Discount_Cost` is computed at the transaction level: the revenue foregone due to discounting

In [ ]:
# ── Step 3.1 — Merge channel attributes onto transactions ─────────────────
df_enriched = df.merge(
    customers[['Customer ID', 'Acquisition_Channel', 'CAC',
                'Acquisition_Campaign', 'LTV_CAC_Ratio']],
    on='Customer ID',
    how='left'
)

# ── Step 3.2 — Add transaction-level discount cost ────────────────────────
# Discount_Cost = revenue foregone = (undiscounted price × discount rate)
# Undiscounted price = Sales / (1 − Discount); handle Discount = 0 with fillna
df_enriched['Discount_Cost'] = (
    df_enriched['Sales'] / (1 - df_enriched['Discount'].replace(0, np.nan))
).fillna(df_enriched['Sales']) * df_enriched['Discount']

print(f"Enriched table: {df_enriched.shape[0]:,} rows × {df_enriched.shape[1]} columns")
print(f"Original columns: {df.shape[1]}  |  Added: {df_enriched.shape[1] - df.shape[1]}")
print(f"\nNew columns: {[c for c in df_enriched.columns if c not in df.columns]}")

In [ ]:
# ── Step 3.3 — Validate ──────────────────────────────────────────────────────
assert len(df_enriched) == len(df), "Row count changed after merge — check join logic"
assert df_enriched[['Acquisition_Channel','CAC','Acquisition_Campaign','LTV_CAC_Ratio',
                     'Discount_Cost']].isnull().sum().sum() == 0, "Nulls in new columns"
assert (df_enriched['Discount_Cost'] >= 0).all(), "Negative discount cost found"

print("✓ Row count preserved after merge:", len(df_enriched))
print("✓ No nulls in enriched columns")
print("✓ Discount_Cost ≥ 0 for all rows")

print("\nDiscount_Cost stats:")
print(df_enriched['Discount_Cost'].describe().round(2))
print(f"\nTotal discount cost across all transactions: ${df_enriched['Discount_Cost'].sum():,.0f}")
print(f"Total profit across all transactions:        ${df_enriched['Profit'].sum():,.0f}")
print(f"Discount-to-profit ratio:                    {df_enriched['Discount_Cost'].sum()/df_enriched['Profit'].sum():.2f}×")

In [ ]:
# ── Step 3.4 — Channel distribution in enriched table ────────────────────────
print("Transactions by acquisition channel:")
print(df_enriched['Acquisition_Channel'].value_counts())

print("\nProfit by acquisition channel:")
print(df_enriched.groupby('Acquisition_Channel')['Profit'].agg(
    ['sum','mean','median']).round(2).sort_values('sum', ascending=False))

---
## Save all three tables

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/raw', exist_ok=True)

# ── Table 1 ──────────────────────────────────────────────────────────────────
customers.to_parquet('../data/processed/customer_acquisition.parquet', index=False)
customers.to_csv('../data/processed/customer_acquisition.csv', index=False)

# ── Table 2 ──────────────────────────────────────────────────────────────────
spend_df.to_parquet('../data/processed/monthly_channel_spend.parquet', index=False)
spend_df.to_csv('../data/processed/monthly_channel_spend.csv', index=False)

# ── Table 3 ──────────────────────────────────────────────────────────────────
df_enriched.to_parquet('../data/processed/superstore_enriched.parquet', index=False)
df_enriched.to_csv('../data/processed/superstore_enriched.csv', index=False)

print("All tables saved to data/processed/:")
print(f"  customer_acquisition    — {len(customers):,} rows × {len(customers.columns)} cols")
print(f"  monthly_channel_spend   — {len(spend_df):,} rows × {len(spend_df.columns)} cols")
print(f"  superstore_enriched     — {len(df_enriched):,} rows × {len(df_enriched.columns)} cols")

---
## Data Dictionary

### `customer_acquisition`
| Column | Type | Description |
|--------|------|-------------|
| `Customer ID` | string | Join key to all other tables |
| `Customer_Name` | string | Customer display name |
| `Segment` | string | Consumer / Corporate / Home Office |
| `Region` | string | East / West / Central / South |
| `First_Order` | date | Date of first transaction |
| `Total_Orders` | int | Distinct orders placed (2014–2017) |
| `Total_Revenue` | float | Sum of Sales across all orders |
| `Total_Profit` | float | Sum of Profit across all orders |
| `Acquisition_Channel` | string | **Synthetic** — channel that acquired this customer |
| `CAC` | float | **Synthetic** — cost to acquire this customer ($) |
| `Acquisition_Campaign` | string | **Synthetic** — specific campaign within the channel |
| `LTV_CAC_Ratio` | float | Total_Profit / CAC — lifetime return per $ spent |

### `monthly_channel_spend`
| Column | Type | Description |
|--------|------|-------------|
| `Period` | string | Year-Month (e.g. "2014-01") |
| `Year` / `Month` / `Quarter` | int | Time dimension breakouts |
| `Channel` | string | Marketing channel |
| `Spend` | float | **Synthetic** — monthly spend ($) |
| `Impressions` | int | **Synthetic** — ad impressions (0 for Email/Organic/Direct) |
| `Clicks` | int | **Synthetic** — link clicks |
| `Conversions` | int | **Synthetic** — attributed conversions |
| `Revenue_Attributed` | float | **Synthetic** — revenue attributed to this channel/month |
| `ROAS` | float | **Synthetic** — Revenue_Attributed / Spend |

### `superstore_enriched`
All original columns from `superstore_clean` plus:
| Column | Type | Description |
|--------|------|-------------|
| `Acquisition_Channel` | string | **Synthetic** — inherited from customer_acquisition |
| `CAC` | float | **Synthetic** — inherited from customer_acquisition |
| `Acquisition_Campaign` | string | **Synthetic** — inherited from customer_acquisition |
| `LTV_CAC_Ratio` | float | **Synthetic** — inherited from customer_acquisition |
| `Discount_Cost` | float | Computed — revenue foregone due to discount on this transaction |

> **All synthetic columns are clearly labelled above.** Any analysis using these columns constitutes a simulation exercise with academic approval, not a claim of real-world measurement.